# 03. Scoring models
Logistic regression on WoE (scorecard) and LightGBM, each with and without EXT_SOURCE_1/2/3.
Bins, WoE and feature selection are fitted on **train**; tuning and comparison use **valid**;
test is not loaded in this notebook (decision 23).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.viz import INK_MUTED, INK_SECONDARY, SOURCE_COLORS, apply_style, save_figure
from src.woe import MISSING, WoETransformer, fit_feature, select_features

MART_PATH = PROJECT_ROOT / "data" / "processed" / "data_mart.parquet"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
RANDOM_STATE = 42

apply_style()
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_rows", 100)

In [ ]:
mart = pd.read_parquet(MART_PATH)
train = mart[mart["sample"] == "train"].reset_index(drop=True)
valid = mart[mart["sample"] == "valid"].reset_index(drop=True)
del mart  # test stays untouched until the final evaluation

# CODE_GENDER is kept in the mart only for the fairness audit (decision 15)
NOT_FEATURES = ["sample", "SK_ID_CURR", "TARGET", "CODE_GENDER"]
FEATURES = [c for c in train.columns if c not in NOT_FEATURES]
EXT_FEATURES = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

assert len(train) == 184_506 and len(valid) == 61_502
print(f"{len(FEATURES)} candidate features; train {len(train):,}, valid {len(valid):,}")

## 3.2 Binning, WoE and Information Value (train)

In [ ]:
woe = WoETransformer(max_bins=20, min_share=0.05).fit(train[FEATURES], train["TARGET"])
iv = woe.iv_table()

# Price of monotonicity: IV of the same numeric feature with bins free to be U-shaped
iv_free = {
    f: fit_feature(train[f], train["TARGET"], monotonic=False).iv
    for f in iv.loc[iv["kind"] == "numeric", "feature"]
}
iv["iv_non_monotonic"] = iv["feature"].map(iv_free)
iv["iv_lost_by_monotonic"] = 1 - iv["iv"] / iv["iv_non_monotonic"]


def iv_strength(v: float) -> str:
    if v < 0.02:
        return "useless"
    if v < 0.1:
        return "weak"
    if v < 0.3:
        return "medium"
    if v < 0.5:
        return "strong"
    return "suspicious"


iv["strength"] = iv["iv"].map(iv_strength)
iv.round(4)

In [ ]:
# Features where forcing monotonic bins costs the most information
iv[iv["iv_lost_by_monotonic"] > 0.15].sort_values("iv_non_monotonic", ascending=False).round(4)

In [ ]:
# Example: full binning table of one feature
woe.binnings_["age_years"].table.round(4)

In [ ]:
PREFIX_TO_SOURCE = {"bureau_": "bureau", "prev_": "previous", "inst_": "installments",
                    "pos_": "pos", "cc_": "credit_card"}


def column_source(col: str) -> str:
    return next((s for p, s in PREFIX_TO_SOURCE.items() if col.startswith(p)), "application")


plot_df = iv.sort_values("iv")
fig, ax = plt.subplots(figsize=(9, 13))
bars = ax.barh(plot_df["feature"], plot_df["iv"],
               color=[SOURCE_COLORS[column_source(f)] for f in plot_df["feature"]], height=0.7)
for bar, v in zip(bars, plot_df["iv"]):
    ax.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
            va="center", fontsize=7, color=INK_SECONDARY)
for threshold, label in [(0.02, "0.02 useless"), (0.1, "0.1 medium"), (0.3, "0.3 strong")]:
    ax.axvline(threshold, color=INK_MUTED, linestyle="--", linewidth=1)
    ax.text(threshold, 1.002, label, transform=ax.get_xaxis_transform(), ha="center",
            va="bottom", fontsize=8, color=INK_MUTED)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=8)
ax.set_xlabel("Information Value (train)")
ax.set_title("Information Value of candidate features", pad=18)
sources = [s for s in SOURCE_COLORS if s in {column_source(f) for f in plot_df["feature"]}]
ax.legend([plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS[s]) for s in sources], sources,
          title="Source", loc="lower right")

save_figure(fig, "03_information_value", FIGURES_DIR)
plt.show()

In [ ]:
# Draft selection: IV >= 0.02, |corr of WoE| <= 0.7, greedy from the highest IV
woe_train = woe.transform(train[FEATURES])
iv_series = iv.set_index("feature")["iv"]

selection_ext = select_features(woe_train, iv_series, min_iv=0.02, max_corr=0.7)
no_ext = [f for f in FEATURES if f not in EXT_FEATURES]
selection_no_ext = select_features(woe_train[no_ext], iv_series[no_ext], min_iv=0.02, max_corr=0.7)

print("Selected with EXT_SOURCE:   ", selection_ext["selected"].sum())
print("Selected without EXT_SOURCE:", selection_no_ext["selected"].sum())
selection_ext.round(4)

## 3.3 Logistic regression on WoE (scorecard)
Final binning: monotonic WoE for every numeric feature except `credit_term`, which is allowed
a U-shape (decision 25: monotonic bins lose 74% of its IV). Selection: IV >= 0.02, |corr| <= 0.7.

In [ ]:
from src.scorecard import fit_logreg_woe, gini, score, scorecard_table

woe = WoETransformer(max_bins=20, min_share=0.05, non_monotonic=("credit_term",))
woe.fit(train[FEATURES], train["TARGET"])
iv_final = woe.iv_table().set_index("feature")["iv"]

woe_train = woe.transform(train[FEATURES])
woe_valid = woe.transform(valid[FEATURES])  # bins learned on train, applied unchanged

NO_EXT = [f for f in FEATURES if f not in EXT_FEATURES]
selected = {
    "with_ext": select_features(woe_train, iv_final, min_iv=0.02, max_corr=0.7),
    "no_ext": select_features(woe_train[NO_EXT], iv_final[NO_EXT], min_iv=0.02, max_corr=0.7),
}
for version, sel in selected.items():
    print(version, "->", sel["selected"].sum(), "features")

In [ ]:
logreg = {}
rows = []
for version, sel in selected.items():
    candidates = sel.loc[sel["selected"], "feature"].tolist()
    model, features, dropped = fit_logreg_woe(woe_train, train["TARGET"], candidates, random_state=RANDOM_STATE)
    logreg[version] = (model, features)
    pd_train = model.predict_proba(woe_train[features])[:, 1]
    pd_valid = model.predict_proba(woe_valid[features])[:, 1]
    rows.append({
        "model": f"logreg_woe_{version}",
        "n_features": len(features),
        "dropped_wrong_sign": ", ".join(dropped) or "-",
        "gini_train": gini(train["TARGET"], pd_train),
        "gini_valid": gini(valid["TARGET"], pd_valid),
    })

results_logreg = pd.DataFrame(rows)
results_logreg["gini_drop"] = results_logreg["gini_train"] - results_logreg["gini_valid"]
results_logreg.round(4)

In [ ]:
# Coefficients of the model with external scores: all negative by construction
model, features = logreg["with_ext"]
coefs = pd.DataFrame({"feature": features, "coef": model.coef_[0], "iv": iv_final[features].values})
coefs.sort_values("coef").round(4)

In [ ]:
# Scorecard: points per bin; 600 points = odds 50:1 (good:bad), +20 points doubles the odds
card = scorecard_table(woe.binnings_, model, features)
card["points"] = card["points"].round(0).astype(int)
card[card["feature"].isin(["EXT_SOURCE_3", "age_years", "credit_term", "OCCUPATION_TYPE"])].round(4)

In [ ]:
from src.viz import BAD_COLOR, GOOD_COLOR

valid_score = score(woe_valid, model, features)
bins_hist = np.arange(np.floor(valid_score.min() / 10) * 10, valid_score.max() + 10, 10)

fig, ax = plt.subplots(figsize=(10, 4.8))
for label, mask, color in [("repaid (TARGET = 0)", valid["TARGET"] == 0, GOOD_COLOR),
                           ("defaulted (TARGET = 1)", valid["TARGET"] == 1, BAD_COLOR)]:
    ax.hist(valid_score[mask], bins=bins_hist, density=True, histtype="step", linewidth=2,
            color=color, label=f"{label}: median {valid_score[mask].median():.0f}")
ax.set_xlabel("Scorecard score (higher = safer)")
ax.set_ylabel("Density")
ax.set_title(f"Score distribution on valid, scorecard with external scores "
             f"(Gini {gini(valid['TARGET'], -valid_score):.3f})")
ax.legend(loc="upper right")

save_figure(fig, "03_score_distribution", FIGURES_DIR)
plt.show()